In [14]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
from itertools import product
import calendar

In [15]:
BASE_DIR = "dataset"

dfs = {}

for file in os.listdir(BASE_DIR):
    if file.endswith(".csv"):
        path = os.path.join(BASE_DIR, file)

        df = pd.read_csv(path)

        name = file.replace(".csv", "")
        dfs[name] = df
dfs.keys()
tables_name = dfs.keys()-{'data_dictionary'}


# EDA 1

In [16]:
table_info = []
for name in tables_name:
    data = dfs[name]
    table_info.append(
        {"Table":name,
         "Rows":data.shape[0],
         "Columns":data.shape[1],
         "missing_value":data.isna().sum().sum(),
         "duplicated data":data.duplicated().sum()
         }
    )
display(pd.DataFrame(table_info))
    

,Table,Rows,Columns,missing_value,duplicated data
0,sales_pipeline,8800,8,5015,0
1,accounts,85,7,70,0
2,products,7,3,0,0
3,sales_teams,35,3,0,0


In [17]:
table_columns_type = []
for name in tables_name:
    data = dfs[name].dtypes
    for column,type in data.items():
        table_columns_type.append(
            {"table":name,
            "column":column,
            "type of data":type,
            "missing values": dfs[name][column].isna().sum(),
            "unique values": dfs[name][column].nunique()}
        )
data_table_info = pd.DataFrame(table_columns_type)
data_table_info

,table,column,type of data,missing values,unique values
0,sales_pipeline,opportunity_id,str,0,8800
1,sales_pipeline,sales_agent,str,0,30
2,sales_pipeline,product,str,0,7
3,sales_pipeline,account,str,337,85
4,sales_pipeline,deal_stage,str,0,4
5,sales_pipeline,engage_date,str,500,421
6,sales_pipeline,close_date,str,2089,306
7,sales_pipeline,close_value,float64,2089,2051
8,accounts,account,str,0,85
9,accounts,sector,str,0,10


# Data Cleaning

## 
Accounts Table

In [18]:
df_accounts = dfs['accounts']
display(data_table_info[data_table_info['table']=="accounts"])
df_accounts.loc[df_accounts['subsidiary_of'].isna(),'subsidiary_of'] = 'Unknown'
columns = (df_accounts.select_dtypes(include=['object','str'])).columns
for column in columns:
    df_accounts[column] = df_accounts[column].str.strip()
display(df_accounts)
display(df_accounts.info())
display(df_accounts.describe())

,table,column,type of data,missing values,unique values
8,accounts,account,str,0,85
9,accounts,sector,str,0,10
10,accounts,year_established,int64,0,35
11,accounts,revenue,float64,0,85
12,accounts,employees,int64,0,85
13,accounts,office_location,str,0,15
14,accounts,subsidiary_of,str,70,7


,account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,Acme Corporation,technolgy,1996,1100.04,2822,United States,Unknown
1,Betasoloin,medical,1999,251.41,495,United States,Unknown
2,Betatech,medical,1986,647.18,1185,Kenya,Unknown
3,Bioholding,medical,2012,587.34,1356,Philipines,Unknown
4,Bioplex,medical,1991,326.82,1016,United States,Unknown
...,...,...,...,...,...,...,...
80,Zathunicon,retail,2010,71.12,144,United States,Unknown
81,Zencorporation,technolgy,2011,40.79,142,China,Unknown
82,Zoomit,entertainment,1992,324.19,978,United States,Unknown
83,Zotware,software,1979,4478.47,13809,United States,Unknown


<class 'pandas.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   account           85 non-null     str    
 1   sector            85 non-null     str    
 2   year_established  85 non-null     int64  
 3   revenue           85 non-null     float64
 4   employees         85 non-null     int64  
 5   office_location   85 non-null     str    
 6   subsidiary_of     85 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 4.8 KB


None

,year_established,revenue,employees
count,85.000000,85.000000,85.000000
mean,1996.105882,1994.632941,4660.823529
std,8.865427,2169.491436,5715.601198
min,1979.000000,4.540000,9.000000
25%,1989.000000,497.110000,1179.000000
50%,1996.000000,1223.720000,2769.000000
75%,2002.000000,2741.370000,5595.000000
max,2017.000000,11698.030000,34288.000000


##
Products Table

In [19]:
df_products = dfs['products']
display(data_table_info[data_table_info['table']=='products'])
columns = (df_products.select_dtypes(include = 'str')).columns
for column in columns:
    df_products[column] = df_products[column].str.strip()
display(df_products)
display(df_products.describe())

,table,column,type of data,missing values,unique values
15,products,product,str,0,7
16,products,series,str,0,3
17,products,sales_price,int64,0,7


,product,series,sales_price
0,GTX Basic,GTX,550
1,GTX Pro,GTX,4821
2,MG Special,MG,55
3,MG Advanced,MG,3393
4,GTX Plus Pro,GTX,5482
5,GTX Plus Basic,GTX,1096
6,GTK 500,GTK,26768


,sales_price
count,7.000000
mean,6023.571429
std,9388.428070
min,55.000000
25%,823.000000
50%,3393.000000
75%,5151.500000
max,26768.000000


##
Sales Pipeline Table

In [20]:
df_sales_pipeline = dfs['sales_pipeline']
display(data_table_info[data_table_info['table']=='sales_pipeline'])
column_str = df_sales_pipeline.select_dtypes(
    include=['object', 'str']
).columns

for column in column_str:
    df_sales_pipeline[column] = df_sales_pipeline[column].str.strip()

date_column = ['close_date','engage_date']
for column in date_column:
    df_sales_pipeline[column] = pd.to_datetime(df_sales_pipeline[column])

df_sales_pipeline.loc[df_sales_pipeline['account'].isna(),'account'] = 'Unidentified'
df_sales_pipeline.info()

df_sales_pipeline.loc[df_sales_pipeline['product']=='GTXPro','product'] = 'GTX Pro'



,table,column,type of data,missing values,unique values
0,sales_pipeline,opportunity_id,str,0,8800
1,sales_pipeline,sales_agent,str,0,30
2,sales_pipeline,product,str,0,7
3,sales_pipeline,account,str,337,85
4,sales_pipeline,deal_stage,str,0,4
5,sales_pipeline,engage_date,str,500,421
6,sales_pipeline,close_date,str,2089,306
7,sales_pipeline,close_value,float64,2089,2051


<class 'pandas.DataFrame'>
RangeIndex: 8800 entries, 0 to 8799
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   opportunity_id  8800 non-null   str           
 1   sales_agent     8800 non-null   str           
 2   product         8800 non-null   str           
 3   account         8800 non-null   str           
 4   deal_stage      8800 non-null   str           
 5   engage_date     8300 non-null   datetime64[us]
 6   close_date      6711 non-null   datetime64[us]
 7   close_value     6711 non-null   float64       
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 550.1 KB


At the Prospecting stage, some records contain account information, while others have missing values. This variation is considered consistent with the nature of prospecting, where potential customers may not yet be fully identified or linked to an existing account.

The missing values are therefore not treated as invalid records but as potentially meaningful indicators of the prospect identification process.

- **Available account value:** The potential customer has been identified and linked to an account.
- **Missing account value:** The potential customer may not yet have been identified or linked to an account.

Based on this interpretation, records with missing account values will be retained to avoid unnecessary data loss. Instead, missing values will be replaced with the category **"Unidentified"** to distinguish them from identified accounts while preserving the records for subsequent analysis.

Missing values in the `engage_date`, `close_date`, and `close_value` were not treated, as opportunities in the Prospecting stage are not expected to have engagement dates, closing dates, or closing values yet.

###
Sales Teams Table

In [21]:
df_sales_teams = dfs['sales_teams']
display(data_table_info[data_table_info['table']=="sales_teams"])
columns = df_sales_teams.columns
for column in columns:
    df_sales_teams[column] = df_sales_teams[column].str.strip()
display(df_sales_teams.head())
display(df_sales_teams.info())
display(df_sales_teams.describe())

,table,column,type of data,missing values,unique values
18,sales_teams,sales_agent,str,0,35
19,sales_teams,manager,str,0,6
20,sales_teams,regional_office,str,0,3


,sales_agent,manager,regional_office
0,Anna Snelling,Dustin Brinkmann,Central
1,Cecily Lampkin,Dustin Brinkmann,Central
2,Versie Hillebrand,Dustin Brinkmann,Central
3,Lajuana Vencill,Dustin Brinkmann,Central
4,Moses Frase,Dustin Brinkmann,Central


<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sales_agent      35 non-null     str  
 1   manager          35 non-null     str  
 2   regional_office  35 non-null     str  
dtypes: str(3)
memory usage: 972.0 bytes


None

,sales_agent,manager,regional_office
count,35,35,35
unique,35,6,3
top,Anna Snelling,Melvin Marxen,East
freq,1,6,12


In [22]:
clean_table = [
    df_accounts,
    df_products,
    df_sales_pipeline,
    df_sales_teams
]

table_names = [
    'accounts',
    'products',
    'sales_pipeline',
    'sales_teams'
]

folder = r'E:\5_Portofolio Data Analyst\5_B2B_Performance_Analysis\clean_dataset'

for name, df in zip(table_names, clean_table):
    df.to_csv(os.path.join(folder, f'{name}.csv'), index=False)